In [1]:
# Import packages
import pandas as pd
import numpy as np
import datetime as dt
from sklearn.svm import LinearSVC, SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.metrics import f1_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import log_loss
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score, roc_curve
from sklearn import metrics
import matplotlib.pyplot as plt
import os


In [2]:
# Set path
print(os.getcwd())
os.chdir('../ml_data/')


/Users/davidal-gurnawi/Documents/data_science/machine_learning/support_vector_machines


In [3]:
# Get the final model output
output_path = 'fraud_data_cleaned.csv'

 - Machine learning model capable of performing linear or nonlinear classification, regression, and even novelty  detection.
 - Shines with small to medium-sized nonlinear datasets, especially for classification tasks.
 - They don't scale well to large datasets.

- Think of an SVM classifier as fitting the widest possible street between two classes (large margin classification)
- Adding more training instances "off the street" will not affect the decision boundary at all.
- They are fully determined (or "supported") by the instances on the edge of these "streets". These are called support vectors.

SVMs are sensitive to the feature scales!

- Strict imposition of all instances to be off the street and on the correct side is called **hard margin** 
\n
-**Two problems:**
- It only works if the data is linearly separable.
- it is sensitive to outliers.

- To avoid these issues, need to use a flexible model.
- Keep the street as large as possible and limit the margin violations (instances that end up in the middle of the street or wrong side)
- Called **soft margin classification**

- Unlike LogisticRegression, LinearSVC doesn't have a predict_proba() method to estiamte the class probabilities.
- If you use SVC Class instead of LinearSVC, and set the probability hyperparamter to True, then the model will fit an extra model at the end of training to map the SVM decision function scores to estiamted probabilities.

In [4]:
# Test train split
df = pd.read_csv(output_path)
df = df[['IsFraud','standardised_amount']]
#df['amount_sq'] = df['standardised_amount']**2
#df['amount_cube'] = df['standardised_amount']**3
#df['amount_sqrt'] = df['standardised_amount']**(1/2)
X_train, X_test, y_train, y_test = train_test_split(df.drop('IsFraud', axis=1),df['IsFraud'], test_size=0.25, random_state=42)

In [5]:
svm_clf = LinearSVC(C=100, random_state=42)
svm_cl_model = svm_clf.fit(X_train, y_train)

In [6]:
# Predictions for X_test
y_pred = svm_cl_model.predict(X_test)

In [7]:
# Scoring
train_score = svm_cl_model.score(X_train, y_train)
print(f"Training Accuracy: {round(train_score*100)}%")
test_score = svm_cl_model.score(X_test, y_test)
print(f"Testing Accuracy: {round(test_score*100)}%")

# F1 Score
f1_value = f1_score(y_true=y_test, y_pred=y_pred, average='weighted')
print(f"F1 Score: {round(f1_value,2)}")
# Recall score
recall_value = recall_score(y_true=y_test, y_pred=y_pred, average='weighted')
print(f"Recall Score: {round(recall_value,2)}")
# Precision Score
precision_value = precision_score(y_true=y_test, y_pred=y_pred, average='weighted')
print(f"Precision Score: {round(precision_value,2)}")
cm = confusion_matrix(y_test, y_pred, labels=svm_cl_model.classes_)
tf_pf = cm[0][0]
print(f"Amount Predicted False where Actual False: {tf_pf}")
tf_pt = cm[0][1]
print(f"Amount Predicted True where Actual False: {tf_pt}")
tt_pf = cm[1][0]
print(f"Amount Predicted False where Actual True: {tt_pf}")
tt_pt = cm[1][1]
print(f"Amount Predicted True where Actual True: {tt_pt}")

Training Accuracy: 99%
Testing Accuracy: 98%
F1 Score: 0.97
Recall Score: 0.98
Precision Score: 0.96
Amount Predicted False where Actual False: 15560
Amount Predicted True where Actual False: 0
Amount Predicted False where Actual True: 292
Amount Predicted True where Actual True: 0


/Users/davidal-gurnawi/Documents/data_science/machine_learning/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [8]:
df.head()

,IsFraud,standardised_amount
0,0,0.112740
1,0,-1.187655
2,0,-1.475318
3,0,-0.714213
4,0,-1.095303


In [ ]:
# Trying some graphical illustrations
fig, axes = plt.subplots(ncols=1, figsize=(20, 10), sharey=True)

df_fraud = df[df['IsFraud']==1]
#print(df_fraud.head())
df_no_fraud = df[df['IsFraud']==0]
#print(df_no_fraud.head())
# First subplot
plt.plot('amount_sqrt','per_100k','yo',data=df_fraud, label="Fraud")
plt.plot(df_no_fraud['amount_sqrt'], df_no_fraud['per_100k'],  "bs", label="Not Fraud")
plt.xlabel("Standardised Amount")
plt.ylabel("Amount Threshold")
plt.legend(loc="upper left")
plt.title("Amount v Threshold")
plt.grid()

plt.show()

OK, so this model is garbage. Let's see what else we can do with SVM

# Polynomial Kernel

Makes it possible to get the same result as if you had added many polynomial features, even with a very high ä
degree, without actually having to add them ---> no combinatorial explosion of the number of features.

In [10]:

# c_hyperparam = np.empty(shape=(1))
# degree_hyperparam = np.empty(shape=(1))
# coef_hyperparam = np.empty(shape=(1))
# train_score_array = np.empty(shape=(1))
# test_score_array = np.empty(shape=(1))
# f1_value_array = np.empty(shape=(1))
# recall_value_array = np.empty(shape=(1))
# precision_value_array = np.empty(shape=(1))
# tf_pf_array = np.empty(shape=(1))
# tf_pt_array = np.empty(shape=(1))
# tt_pf_array = np.empty(shape=(1))
# tt_pt_array = np.empty(shape=(1))

svm_clf = SVC(
    C=1,
    kernel='poly', # {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'}
    degree=2, # Only for poly, ignored by other kernels
    coef0=1, # Hyperparameter coef0 controls how much the model is influenced by hig-degree terms versus low-degree terms.
    shrinking=True,
    tol = 1e-4,
    cache_size = 800, #In Mb
    class_weight='balanced',
    verbose=False,
    max_iter=-1, # -1 no limit
    decision_function_shape='ovr',
    random_state=42)


svm_cl_model = svm_clf.fit(X_train, y_train)
# Predictions for X_test
y_pred = svm_cl_model.predict(X_test)
# Scoring
train_score = svm_cl_model.score(X_train, y_train)
print(f"Training Accuracy: {round(train_score*100)}%")
test_score = svm_cl_model.score(X_test, y_test)
print(f"Testing Accuracy: {round(test_score*100)}%")

# F1 Score
f1_value = f1_score(y_true=y_test, y_pred=y_pred, average='weighted')
print(f"F1 Score: {round(f1_value,2)}")
# Recall score
recall_value = recall_score(y_true=y_test, y_pred=y_pred, average='weighted')
print(f"Recall Score: {round(recall_value,2)}")
# Precision Score
precision_value = precision_score(y_true=y_test, y_pred=y_pred, average='weighted')
print(f"Precision Score: {round(precision_value,2)}")
cm = confusion_matrix(y_test, y_pred, labels=svm_cl_model.classes_)
tf_pf = cm[0][0]
print(f"Amount Predicted False where Actual False: {tf_pf}")
tf_pt = cm[0][1]
print(f"Amount Predicted True where Actual False: {tf_pt}")
tt_pf = cm[1][0]
print(f"Amount Predicted False where Actual True: {tt_pf}")
tt_pt = cm[1][1]
print(f"Amount Predicted True where Actual True: {tt_pt}")

# c_hyperparam = np.append(arr=c_hyperparam, values=np.asarray([i]))
# degree_hyperparam = np.append(arr=degree_hyperparam, values=np.asarray([j]))
# coef_hyperparam = np.append(arr=coef_hyperparam, values=np.asarray([k]))
# train_score_array = np.append(arr=train_score_array, values=np.asarray([train_score]))
# test_score_array =np.append(arr=test_score_array, values=np.asarray([test_score]))
# f1_value_array = np.append(arr=f1_score, values=np.asarray([f1_value]))
# recall_value_array = np.append(arr=recall_value_array, values=np.asarray([recall_value]))
# precision_value_array = np.append(arr=precision_value_array, values=np.asarray([precision_value]))
# tf_pf_array = np.append(arr=tf_pf_array, values=np.asarray([tf_pf]))
# tf_pt_array = np.append(arr=tf_pt_array, values=np.asarray([tf_pt]))
# tt_pf_array = np.append(arr=tt_pf_array, values=np.asarray([tt_pf]))
# tt_pt_array = np.append(arr=tt_pt_array, values=np.asarray([tt_pt]))



Training Accuracy: 71%
Testing Accuracy: 70%
F1 Score: 0.81
Recall Score: 0.7
Precision Score: 0.96
Amount Predicted False where Actual False: 11100
Amount Predicted True where Actual False: 4460
Amount Predicted False where Actual True: 219
Amount Predicted True where Actual True: 73
